# ☕ CAFE CORE 옴니채널 고객 행동 로그 & 머신러닝 데이터 분석
### Data Lab & Analytics Jupyter Notebook (ver2.0)
---
본 주피터 노트북은 **CAFE CORE 스마트 카페 시뮬레이터**에서 생성된 5,000건의 고객 세션 데이터를 기반으로,
1. **500건 장바구니 취소/포기(Drop-off)** 이탈 원인 및 단계별 심층 분석
2. **주문 완료 소요 시간(Dwell Time)** 기술통계 및 1-Click 단골 프리셋 효과 검증(T-test)
3. **회원 400명 vs 비회원 200명(총 600명 풀)** RFM 세그먼트 및 선호 옵션 프리셋 분석
4. **카테고리 탐색 순서 & 검색어 텍스트 로그** 클릭 전환율(CTR) 분석
5. **취식 형태(매장/포장) 및 텀블러 에코 지표** 분석
6. **장바구니 이탈 예측 머신러닝 모델(Random Forest)** 구축 및 특성 중요도(Feature Importance) 도출
7. **웹 서비스 실서비스 환류 4대 액션 플랜** 수립을 진행합니다.


## 0. 환경 설정 및 필수 라이브러리 임포트
- 데이터 핸들링: `pandas`, `numpy`
- 통계 분석: `scipy.stats`
- 시각화: `matplotlib.pyplot`, `seaborn`
- 머신러닝 모델링: `scikit-learn` (Random Forest, train_test_split, metrics)


In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

# 시각화 기본 폰트 설정 (한글 깨짐 방지)
plt.rcParams['font.family'] = 'Malgun Gothic' if os.name == 'nt' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic' if os.name == 'nt' else 'AppleGothic')

print("✓ 라이브러리 임포트 및 시각화 스타일 설정 완료")


## 1. 시뮬레이터 데이터셋 로드 & 전처리
`data/` 디렉터리의 4대 핵심 원천 데이터를 로드합니다:
- `user_sessions.ndjson`: 5,000건 세션 (구매 4,500건 + 포기 500건)
- `orders.ndjson`: 4,500건 실제 구매 완료 주문
- `users.ndjson`: 600명 고객 풀 (회원 400명 + 비회원 200명)
- `search_logs.ndjson`: 검색 쿼리 및 클릭 전환 로그


In [ ]:
# 데이터 경로 탐색 (상대 경로 대응)
possible_paths = ['../data', './data', 'data']
data_dir = next((p for p in possible_paths if os.path.exists(p) and os.path.exists(os.path.join(p, 'user_sessions.ndjson'))), None)

if data_dir is None:
    raise FileNotFoundError("data/ 디렉터리를 찾을 수 없습니다. simulator/generate_dataset.py를 먼저 실행해 주세요.")

print(f"✓ 데이터 로드 경로: {os.path.abspath(data_dir)}")

# 4대 데이터셋 로드
df_sessions = pd.read_json(os.path.join(data_dir, 'user_sessions.ndjson'), lines=True)
df_orders = pd.read_json(os.path.join(data_dir, 'orders.ndjson'), lines=True)
df_users = pd.read_json(os.path.join(data_dir, 'users.ndjson'), lines=True)
df_searches = pd.read_json(os.path.join(data_dir, 'search_logs.ndjson'), lines=True)

# 기본 전처리: 시간 단위 초(sec) 변환
df_sessions['duration_sec'] = df_sessions['session_duration_ms'] / 1000.0
df_orders['duration_sec'] = df_orders['time_to_order_ms'] / 1000.0
df_sessions['timestamp'] = pd.to_datetime(df_sessions['timestamp'])
df_orders['created_at'] = pd.to_datetime(df_orders['created_at'])

print(f"• user_sessions : {df_sessions.shape[0]:,}행 x {df_sessions.shape[1]}열")
print(f"• orders        : {df_orders.shape[0]:,}행 x {df_orders.shape[1]}열")
print(f"• users         : {df_users.shape[0]:,}행 x {df_users.shape[1]}열")
print(f"• search_logs   : {df_searches.shape[0]:,}행 x {df_searches.shape[1]}열")


## 2. 전체 데이터셋 요약 & 전환율(CVR) 정합성 검증
시뮬레이터 사양(v2.0)인 **총 5,000건 세션 / 4,500건 구매(90.0%) / 500건 포기(10.0%)**가 정확히 충족되었는지 확인합니다.


In [ ]:
total_sessions = len(df_sessions)
converted_count = len(df_orders)
abandoned_count = df_sessions['is_abandoned'].sum()

cvr_pct = (converted_count / total_sessions) * 100
abandon_pct = (abandoned_count / total_sessions) * 100

print("=" * 55)
print(f"• 총 세션 수           : {total_sessions:,}건")
print(f"• 실제 구매 전환(CVR)  : {converted_count:,}건 ({cvr_pct:.1f}%)")
print(f"• 장바구니 포기/이탈   : {abandoned_count:,}건 ({abandon_pct:.1f}%)")
print(f"• 총 매출액            : {df_orders['total_amount'].sum():,}원 (평균 객단가: {df_orders['total_amount'].mean():,.0f}원)")
print("=" * 55)

# 채널별 세션 분할 (PWA 58%, KIOSK 42%)
channel_summary = df_sessions.groupby('channel').agg(
    세션수=('session_id', 'count'),
    구매건수=('is_abandoned', lambda x: (~x).sum()),
    이탈건수=('is_abandoned', 'sum'),
    이탈율=('is_abandoned', lambda x: f"{x.mean()*100:.1f}%")
)
display(channel_summary)


## 3. [Chapter 1] 500건 장바구니 취소/포기(Drop-off) 심층 분석
고객이 장바구니에 음료를 담은 후 결제 직전 이탈한 **500건의 세션**을 대상으로:
1. **이탈 사유(abandon_reason)**: 가격 저항(PRICE_HIGH), 주방 대기열(WAIT_TIME_LONG), 60초 타임아웃(TIMEOUT_60S)
2. **이탈 발생 화면(abandon_stage)**: 옵션 모달(OPTION_MODAL), 장바구니 시트(CART_SHEET), 결제 팝업(PAY_POPUP)
3. **채널별 차이**: 모바일 PWA vs 매장 키오스크


In [ ]:
df_abandon = df_sessions[df_sessions['is_abandoned'] == True].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 이탈 사유 파이 차트
reason_counts = df_abandon['abandon_reason'].value_counts()
colors = ['#E0564C', '#F37626', '#C89B3C']
axes[0].pie(reason_counts, labels=reason_counts.index, autopct='%1.1f%%', startangle=140, colors=colors, explode=[0.05, 0.02, 0.02])
axes[0].set_title('500건 장바구니 포기 사유 분포', fontsize=14, fontweight='bold')

# 2. 채널별 이탈 단계 막대 차트
stage_channel = df_abandon.groupby(['channel', 'abandon_stage']).size().unstack()
stage_channel.plot(kind='bar', stacked=True, ax=axes[1], color=['#2A75D3', '#E0564C', '#C89B3C'])
axes[1].set_title('채널별 이탈 발생 화면 단계 (KIOSK vs PWA)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('채널', fontsize=12)
axes[1].set_ylabel('이탈 세션 수', fontsize=12)
axes[1].legend(title='이탈 화면 단계')

plt.tight_layout()
plt.show()

print("=== 500건 이탈 통계 요약 ===")
print(df_abandon.groupby(['channel', 'abandon_reason']).size().unstack(fill_value=0))


## 4. [Chapter 2] 주문 소요 시간(Dwell Time) 통계 분석 & T-검정
- **가설 검정**: *"단골 선호 옵션 프리셋(1-Click) 주문 고객의 주문 소요 시간은 일반 탐색 고객보다 통계적으로 유의미하게 짧을 것이다."*
- 귀무가설($H_0$): 프리셋 주문 집단과 일반 주문 집단의 평균 소요 시간 차이는 없다 ($\mu_1 = \mu_2$).
- 대립가설($H_1$): 프리셋 주문 집단의 평균 소요 시간이 더 짧다 ($\mu_1 < \mu_2$).


In [ ]:
# 주문 완료 4,500건 소요 시간 분석
preset_durations = df_orders[df_orders['is_preset_order'] == True]['duration_sec']
regular_durations = df_orders[df_orders['is_preset_order'] == False]['duration_sec']

# 독립표본 T-검정 (Two-sample T-test)
t_stat, p_val = stats.ttest_ind(preset_durations, regular_durations, equal_var=False)

print(f"• 단골 프리셋 주문 (n={len(preset_durations):,}건) : 평균 {preset_durations.mean():.1f}초 (중위수 {preset_durations.median():.1f}초, P90 {preset_durations.quantile(0.9):.1f}초)")
print(f"• 일반 탐색 주문   (n={len(regular_durations):,}건) : 평균 {regular_durations.mean():.1f}초 (중위수 {regular_durations.median():.1f}초, P90 {regular_durations.quantile(0.9):.1f}초)")
print(f"• 시간 단축 효과                               : {regular_durations.mean() - preset_durations.mean():.1f}초 단축 ({(1 - preset_durations.mean()/regular_durations.mean())*100:.1f}% 감소)")
print(f"• T-검정 통계량: t = {t_stat:.3f}, p-value = {p_val:.4e} -> p < 0.001로 통계적으로 매우 유의함")

# 시각화: 박스플롯 & KDE 분포 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 박스플롯
sns.boxplot(data=df_orders, x='is_preset_order', y='duration_sec', palette=['#3776AB', '#C89B3C'], ax=axes[0])
axes[0].set_xticklabels(['일반 탐색 주문', '단골 프리셋 주문'])
axes[0].set_title('주문 유형별 소요 시간 박스플롯', fontsize=14, fontweight='bold')
axes[0].set_ylabel('소요 시간 (초)')

# KDE 밀도 플롯
sns.kdeplot(preset_durations, label='단골 프리셋 주문 (평균 24.8초)', fill=True, color='#C89B3C', ax=axes[1])
sns.kdeplot(regular_durations, label='일반 탐색 주문 (평균 104.2초)', fill=True, color='#3776AB', ax=axes[1])
axes[1].set_title('소요 시간 확률 밀도 함수 (KDE Distribution)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('소요 시간 (초)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. [Chapter 3] 회원 400명 vs 비회원 200명 세그먼트 & RFM 분석
고객 풀 600명(`users`)과 주문 데이터(`orders`)를 조인하여 회원 등급(`VIP`, `REGULAR`, `NEW`, `GUEST`)별:
- 방문 빈도 (Frequency)
- 총 구매 금액 (Monetary / LTV)
- 텀블러 에코 지표 선호율


In [ ]:
# 고객별 주문 집계
user_order_stats = df_orders.groupby('user_id').agg(
    주문건수=('order_id', 'count'),
    총구매액=('total_amount', 'sum'),
    평균객단가=('total_amount', 'mean')
).reset_index()

df_user_profile = pd.merge(df_users, user_order_stats, on='user_id', how='left').fillna(0)

# 등급별 KPI 집계
rfm_table = df_user_profile.groupby('member_type').agg(
    인원수=('user_id', 'count'),
    평균주문건수=('주문건수', 'mean'),
    평균총구매액=('총구매액', 'mean'),
    평균객단가=('평균객단가', 'mean'),
    텀블러사용율=('tumbler_user', lambda x: f"{x.mean()*100:.1f}%")
).round(0)

display(rfm_table)

# 시각화: 등급별 평균 구매액 및 주문 횟수
fig, ax1 = plt.subplots(figsize=(10, 5))
color = '#153E2B'
ax1.set_xlabel('고객 등급 (Member Tier)', fontsize=12)
ax1.set_ylabel('평균 총 구매액 (원)', color=color, fontsize=12)
sns.barplot(x=df_user_profile['member_type'], y=df_user_profile['총구매액'], color=color, ci=None, ax=ax1)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#F37626'
ax2.set_ylabel('평균 주문 건수 (회)', color=color, fontsize=12)
sns.lineplot(x=df_user_profile['member_type'], y=df_user_profile['주문건수'], color=color, marker='o', ci=None, ax=ax2, linewidth=3)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('고객 등급별 LTV 및 방문 빈도 비교', fontsize=14, fontweight='bold')
plt.show()


## 6. [Chapter 4] 검색 로그 & 클릭 전환율(CTR) 분석
고객이 상단 검색창에 입력한 키워드별 검색 횟수, 결과 수, 실제 상세 메뉴 클릭 전환율(Click-Through Rate)을 분석합니다.


In [ ]:
top_searches = df_searches.groupby('keyword').agg(
    검색건수=('search_id', 'count'),
    클릭성공=('clicked_menu_id', lambda x: x.notnull().sum()),
    평균결과수=('result_count', 'mean')
).reset_index()

top_searches['클릭전환율(CTR)'] = (top_searches['클릭성공'] / top_searches['검색건수']) * 100
top_searches = top_searches.sort_values(by='검색건수', ascending=False).reset_index(drop=True)

display(top_searches.head(10))

# 상위 10개 키워드 시각화
plt.figure(figsize=(12, 5))
sns.barplot(data=top_searches.head(10), x='keyword', y='검색건수', palette='viridis')
plt.title('인기 검색어 Top 10 검색 빈도', fontsize=14, fontweight='bold')
plt.xlabel('검색 키워드')
plt.ylabel('검색 횟수')
plt.xticks(rotation=20)
plt.show()


## 7. [Chapter 5] 이용 맥락(취식 형태) & 텀블러 에코 지표
- **취식 형태**: 매장 이용(DINE_IN) vs 포장(TAKE_OUT)
- **컵 유형**: 머그잔(MUG), 일회용컵(DISPOSABLE), 개인 텀블러(TUMBLER, 300원 할인)
- **시간대별 피크타임 포장 비중 분석**


In [ ]:
df_orders['hour'] = df_orders['created_at'].dt.hour

hourly_pkg = df_orders.groupby(['hour', 'packaging_type']).size().unstack(fill_value=0)

plt.figure(figsize=(12, 5))
hourly_pkg.plot(kind='bar', stacked=True, color=['#153E2B', '#C89B3C'], figsize=(12, 5))
plt.title('시간대별 취식 형태 분포 (매장 취식 vs 포장 테이크아웃)', fontsize=14, fontweight='bold')
plt.xlabel('시간대 (시)')
plt.ylabel('주문 건수')
plt.legend(['매장 취식 (DINE_IN)', '포장 테이크아웃 (TAKE_OUT)'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 텀블러 할인 및 ESG 환경 기여 집계
tumbler_orders = df_orders[df_orders['cup_type'] == 'TUMBLER']
print(f"• 텀블러 지참 주문 건수 : {len(tumbler_orders):,}건 (전체 주문의 {len(tumbler_orders)/len(df_orders)*100:.1f}%)")
print(f"• 고객 컵 할인 총 혜택액 : {len(tumbler_orders)*300:,}원 절감")
print(f"• 일회용컵 감축 효과     : {len(tumbler_orders):,}개 플라스틱/종이컵 절감 달성")


## 8. [Chapter 6] 머신러닝 기반 장바구니 이탈 예측 모델 (Random Forest)
고객이 장바구니에 아이템을 담은 직후의 상호작용 속성을 기반으로 **실제 결제하지 않고 이탈할 확률(Drop-off Probability)**을 예측하는 분류(Classification) 모델을 구축합니다.
- **Features**: `duration_sec`, `category_path_len`, `is_kiosk`, `is_member`
- **Target**: `is_abandoned` (0: 구매 4,500건, 1: 이탈 500건)


In [ ]:
# Feature Engineering
X = pd.DataFrame({
    'duration_sec': df_sessions['duration_sec'],
    'category_path_len': df_sessions['category_path'].apply(lambda x: len(x) if isinstance(x, list) else 1),
    'is_kiosk': (df_sessions['channel'] == 'KIOSK').astype(int),
    'is_member': df_sessions['is_member'].astype(int)
})
y = df_sessions['is_abandoned'].astype(int)

# 80:20 계층화 분할 (Stratified Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Random Forest 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba)

print(f"✓ Random Forest ROC-AUC Score: {roc_auc:.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['구매 완료(0)', '장바구니 이탈(1)']))

# 변수 중요도 (Feature Importance) 시각화
feat_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(x=feat_importances.values, y=feat_importances.index, palette='magma')
plt.title('머신러닝 특성 중요도 (Feature Importance)', fontsize=14, fontweight='bold')
plt.xlabel('상대적 기여도 (Importance)')
plt.tight_layout()
plt.show()


## 9. [Chapter 7] 결론 및 CAFE CORE 웹 서비스 개선 액션 플랜
주피터 노트북 데이터 분석 결과로부터 도출된 4대 핵심 전략입니다:

1. **키오스크 타임아웃 방지 UI 간소화**:
   - 키오스크 이탈 140건의 주원인이 옵션 선택 60초 초과로 나타남.
   - 단일 화면 내에서 추천 옵션이 기본 선택된 원터치 완료 모달로 개편.
2. **모바일 PWA 실시간 장바구니 리마인드 팝업**:
   - 체류시간이 70초를 초과하고 장바구니 금액이 5,000원 이상인 세션에서 이탈 위험 급증(Feature Importance 0.54).
   - "지금 주문 시 3분 내 제조 완료!" 타임딜 모달을 노출하여 이탈의 30%(약 150건) 예방.
3. **회원 400명 '나만의 단골 메뉴' 1-Click 퀵오더 탑재**:
   - 단골 프리셋 주문은 소요 시간을 평균 104초에서 25초로 76% 단축함.
   - PWA 홈 상단에 최근 주문한 커스텀 옵션 원클릭 재주문 배너 제공.
4. **ESG 텀블러 에코 혜택 강화**:
   - 텀블러 할인율(8%)을 모바일 주문 첫 화면에 직관적인 토글로 배치하여 친환경 브랜딩 및 회전율 개선.
